# Problem 2: Diffusion Model Explorations

Use the class notation: drift rate $v$, boundary separation $a$, starting point $\beta$, and non-decision time $\tau$.

In [1]:
import numpy as np
import pandas as pd

N = 2000
dt = 0.001
max_t = 20.0

v0 = 1.0
a0 = 2.0
beta0 = 0.45
tau0 = 0.30

In [2]:
def simulate_ddm(n, v, a, beta, tau, seed=42):
    np.random.seed(seed)
    x = np.full(n, beta * a)
    rt = np.zeros(n)
    choice = np.zeros(n, dtype=int)
    active = np.ones(n, dtype=bool)

    for _ in range(int(max_t / dt)):
        idx = np.where(active)[0]
        if len(idx) == 0:
            break

        x[idx] = x[idx] + v * dt + np.random.normal(0, np.sqrt(dt), len(idx))
        rt[idx] = rt[idx] + dt

        upper = idx[x[idx] >= a]
        lower = idx[x[idx] <= 0]

        choice[upper] = 1
        choice[lower] = -1
        active[upper] = False
        active[lower] = False

    upper_rt = rt[choice == 1] + tau
    lower_rt = rt[choice == -1] + tau

    return upper_rt, lower_rt

In [3]:
v_values = np.linspace(0.5, 1.5, 25)
drift_rows = []

for v in v_values:
    upper_rt, lower_rt = simulate_ddm(N, v, a0, beta0, tau0)
    drift_rows.append(
        {
            "v": v,
            "n_upper": len(upper_rt),
            "n_lower": len(lower_rt),
            "mean_upper": upper_rt.mean(),
            "mean_lower": lower_rt.mean(),
            "mean_diff": upper_rt.mean() - lower_rt.mean(),
        }
    )

drift_df = pd.DataFrame(drift_rows)
print("mean difference range:", round(drift_df["mean_diff"].min(), 3), "to", round(drift_df["mean_diff"].max(), 3), "seconds")
print("smallest lower-bound count:", int(drift_df["n_lower"].min()))

mean difference range: 0.023 to 0.189 seconds
smallest lower-bound count: 153


In [4]:
parameter_ranges = {
    "v": np.linspace(0.5, 1.5, 25),
    "a": np.linspace(1.4, 2.6, 25),
    "beta": np.linspace(0.30, 0.60, 25),
    "tau": np.linspace(0.10, 0.50, 25),
}

summary_rows = []

for parameter in parameter_ranges:
    rows = []

    for value in parameter_ranges[parameter]:
        v = v0
        a = a0
        beta = beta0
        tau = tau0

        if parameter == "v":
            v = value
        if parameter == "a":
            a = value
        if parameter == "beta":
            beta = value
        if parameter == "tau":
            tau = value

        upper_rt, lower_rt = simulate_ddm(N, v, a, beta, tau)
        rows.append(
            {
                "value": value,
                "mean_upper": upper_rt.mean(),
                "mean_lower": lower_rt.mean(),
                "sd_upper": upper_rt.std(ddof=1),
                "sd_lower": lower_rt.std(ddof=1),
                "mean_diff": upper_rt.mean() - lower_rt.mean(),
                "min_count": min(len(upper_rt), len(lower_rt)),
            }
        )

    temp = pd.DataFrame(rows)
    summary_rows.append(
        {
            "parameter": parameter,
            "value_min": temp["value"].iloc[0],
            "value_max": temp["value"].iloc[-1],
            "mean_upper_start": temp["mean_upper"].iloc[0],
            "mean_upper_end": temp["mean_upper"].iloc[-1],
            "mean_lower_start": temp["mean_lower"].iloc[0],
            "mean_lower_end": temp["mean_lower"].iloc[-1],
            "sd_upper_start": temp["sd_upper"].iloc[0],
            "sd_upper_end": temp["sd_upper"].iloc[-1],
            "sd_lower_start": temp["sd_lower"].iloc[0],
            "sd_lower_end": temp["sd_lower"].iloc[-1],
            "mean_diff_start": temp["mean_diff"].iloc[0],
            "mean_diff_end": temp["mean_diff"].iloc[-1],
            "min_count": int(temp["min_count"].min()),
        }
    )

summary_df = pd.DataFrame(summary_rows)
print(summary_df.round(3).to_string(index=False))

parameter  value_min  value_max  mean_upper_start  mean_upper_end  mean_lower_start  mean_lower_end  sd_upper_start  sd_upper_end  sd_lower_start  sd_lower_end  mean_diff_start  mean_diff_end  min_count
        v        0.5        1.5             1.336           0.966             1.166           0.912           0.762         0.456           0.741         0.493            0.170          0.054        153
        a        1.4        2.6             0.772           1.516             0.725           1.354           0.359         0.816           0.373         0.739            0.046          0.161        171
     beta        0.3        0.6             1.295           0.972             0.816           1.226           0.639         0.608           0.498         0.634            0.479         -0.254        162
      tau        0.1        0.5             0.957           1.357             0.815           1.215           0.627         0.627           0.553         0.553            0.142          0.